In [55]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import desc, row_number
from pyspark.sql.window import Window

In [17]:
spark = SparkSession.builder.appName("cybersecurity_lakehouse").getOrCreate()

In [20]:
df = spark.read.csv("../data/raw/security_logs.csv", header = True, inferSchema=True)

In [25]:
df.show()

+-------------------+-------+---------------+--------------+-----------+-------+--------+
|         event_time|user_id|     ip_address|    event_type|device_type|country|severity|
+-------------------+-------+---------------+--------------+-----------+-------+--------+
|2026-04-27 14:12:03|    263| 161.129.145.14| SUSPICIOUS_IP|      Linux|    USA|  MEDIUM|
|2026-03-02 14:11:14|    632|104.173.250.193|FIREWALL_BLOCK|      Linux|  India|  MEDIUM|
|2026-01-13 02:28:48|    430| 26.119.135.185| LOGIN_SUCCESS|      Linux|Germany|  MEDIUM|
|2026-01-14 09:38:50|    403|  173.145.5.120| MALWARE_ALERT|      MacOS|    USA|     LOW|
|2026-03-13 03:30:12|    970|  92.25.167.250| MALWARE_ALERT|    Windows|  India|     LOW|
|2026-02-04 23:31:24|    254|  200.1.214.247|FIREWALL_BLOCK|    Windows|  India|    HIGH|
|2026-05-02 20:54:43|    477| 129.197.38.156|PASSWORD_RESET|    Windows|  India|  MEDIUM|
|2026-01-03 00:21:23|    293|   207.41.54.69| SUSPICIOUS_IP|      MacOS|Germany|    HIGH|
|2026-04-0

In [29]:
df.select('user_id', 'ip_address', 'event_type').show()

+-------+---------------+--------------+
|user_id|     ip_address|    event_type|
+-------+---------------+--------------+
|    263| 161.129.145.14| SUSPICIOUS_IP|
|    632|104.173.250.193|FIREWALL_BLOCK|
|    430| 26.119.135.185| LOGIN_SUCCESS|
|    403|  173.145.5.120| MALWARE_ALERT|
|    970|  92.25.167.250| MALWARE_ALERT|
|    254|  200.1.214.247|FIREWALL_BLOCK|
|    477| 129.197.38.156|PASSWORD_RESET|
|    293|   207.41.54.69| SUSPICIOUS_IP|
|    958|    2.29.115.91|PASSWORD_RESET|
|    481|  171.247.64.31| MALWARE_ALERT|
|    708|   20.133.91.54|PASSWORD_RESET|
|    373|  210.47.115.65|PASSWORD_RESET|
|    614|154.126.156.102| SUSPICIOUS_IP|
|    765| 222.10.156.170| SUSPICIOUS_IP|
|    836|171.204.130.136|FIREWALL_BLOCK|
|    848|   55.66.76.121| MALWARE_ALERT|
|    213|    5.253.61.72|  FAILED_LOGIN|
|    664| 164.177.54.164| SUSPICIOUS_IP|
|    147|    33.43.59.91|  FAILED_LOGIN|
|    514|   8.38.189.160|  FAILED_LOGIN|
+-------+---------------+--------------+
only showing top

In [44]:
df.filter(('device_type == "Linux"') and ('severity == "LOW"')).show()

+-------------------+-------+---------------+--------------+-----------+-------+--------+
|         event_time|user_id|     ip_address|    event_type|device_type|country|severity|
+-------------------+-------+---------------+--------------+-----------+-------+--------+
|2026-01-14 09:38:50|    403|  173.145.5.120| MALWARE_ALERT|      MacOS|    USA|     LOW|
|2026-03-13 03:30:12|    970|  92.25.167.250| MALWARE_ALERT|    Windows|  India|     LOW|
|2026-05-17 20:58:58|    481|  171.247.64.31| MALWARE_ALERT|      MacOS|  India|     LOW|
|2026-02-16 16:51:12|    708|   20.133.91.54|PASSWORD_RESET|      Linux| Canada|     LOW|
|2026-02-05 19:02:32|    614|154.126.156.102| SUSPICIOUS_IP|    Windows|  India|     LOW|
|2026-04-09 17:24:41|    168|171.223.146.109| SUSPICIOUS_IP|    Windows| Canada|     LOW|
|2026-01-23 08:19:20|    575|178.177.116.210| LOGIN_SUCCESS|      Linux|Germany|     LOW|
|2026-02-02 12:35:21|    922|  85.172.41.207| SUSPICIOUS_IP|      MacOS|Germany|     LOW|
|2026-02-2

In [38]:
df.filter(df['country'] == 'India').show()

+-------------------+-------+---------------+--------------+-----------+-------+--------+
|         event_time|user_id|     ip_address|    event_type|device_type|country|severity|
+-------------------+-------+---------------+--------------+-----------+-------+--------+
|2026-03-02 14:11:14|    632|104.173.250.193|FIREWALL_BLOCK|      Linux|  India|  MEDIUM|
|2026-03-13 03:30:12|    970|  92.25.167.250| MALWARE_ALERT|    Windows|  India|     LOW|
|2026-02-04 23:31:24|    254|  200.1.214.247|FIREWALL_BLOCK|    Windows|  India|    HIGH|
|2026-05-02 20:54:43|    477| 129.197.38.156|PASSWORD_RESET|    Windows|  India|  MEDIUM|
|2026-05-17 20:58:58|    481|  171.247.64.31| MALWARE_ALERT|      MacOS|  India|     LOW|
|2026-02-05 19:02:32|    614|154.126.156.102| SUSPICIOUS_IP|    Windows|  India|     LOW|
|2026-05-18 03:12:36|    848|   55.66.76.121| MALWARE_ALERT|      MacOS|  India|    HIGH|
|2026-04-28 08:08:49|    514|   8.38.189.160|  FAILED_LOGIN|      MacOS|  India|  MEDIUM|
|2026-02-1

In [42]:
df.filter((df['country'] == 'India') | (df['country'] == 'Germany')).show()

+-------------------+-------+---------------+--------------+-----------+-------+--------+
|         event_time|user_id|     ip_address|    event_type|device_type|country|severity|
+-------------------+-------+---------------+--------------+-----------+-------+--------+
|2026-03-02 14:11:14|    632|104.173.250.193|FIREWALL_BLOCK|      Linux|  India|  MEDIUM|
|2026-01-13 02:28:48|    430| 26.119.135.185| LOGIN_SUCCESS|      Linux|Germany|  MEDIUM|
|2026-03-13 03:30:12|    970|  92.25.167.250| MALWARE_ALERT|    Windows|  India|     LOW|
|2026-02-04 23:31:24|    254|  200.1.214.247|FIREWALL_BLOCK|    Windows|  India|    HIGH|
|2026-05-02 20:54:43|    477| 129.197.38.156|PASSWORD_RESET|    Windows|  India|  MEDIUM|
|2026-01-03 00:21:23|    293|   207.41.54.69| SUSPICIOUS_IP|      MacOS|Germany|    HIGH|
|2026-05-17 20:58:58|    481|  171.247.64.31| MALWARE_ALERT|      MacOS|  India|     LOW|
|2026-02-05 19:02:32|    614|154.126.156.102| SUSPICIOUS_IP|    Windows|  India|     LOW|
|2026-05-1

In [45]:
df.filter('severity == "HIGH"').count()

3372

In [48]:
df.filter('severity == "HIGH"').collect()

[Row(event_time=datetime.datetime(2026, 2, 4, 23, 31, 24), user_id=254, ip_address='200.1.214.247', event_type='FIREWALL_BLOCK', device_type='Windows', country='India', severity='HIGH'),
 Row(event_time=datetime.datetime(2026, 1, 3, 0, 21, 23), user_id=293, ip_address='207.41.54.69', event_type='SUSPICIOUS_IP', device_type='MacOS', country='Germany', severity='HIGH'),
 Row(event_time=datetime.datetime(2026, 1, 7, 20, 27, 15), user_id=373, ip_address='210.47.115.65', event_type='PASSWORD_RESET', device_type='Windows', country='Canada', severity='HIGH'),
 Row(event_time=datetime.datetime(2026, 1, 16, 4, 6, 45), user_id=836, ip_address='171.204.130.136', event_type='FIREWALL_BLOCK', device_type='Windows', country='Canada', severity='HIGH'),
 Row(event_time=datetime.datetime(2026, 5, 18, 3, 12, 36), user_id=848, ip_address='55.66.76.121', event_type='MALWARE_ALERT', device_type='MacOS', country='India', severity='HIGH'),
 Row(event_time=datetime.datetime(2026, 3, 8, 15, 9, 39), user_id=213

In [57]:
window_spec = Window.orderBy('event_time')
df.withColumn('Rank', row_number().over(window_spec)).show()

+-------------------+-------+---------------+--------------+-----------+-------+--------+----+
|         event_time|user_id|     ip_address|    event_type|device_type|country|severity|Rank|
+-------------------+-------+---------------+--------------+-----------+-------+--------+----+
|2026-01-01 00:49:32|    519| 139.214.81.221| LOGIN_SUCCESS|      Linux|    USA|    HIGH|   1|
|2026-01-01 00:50:13|    212|  10.217.186.35| LOGIN_SUCCESS|      Linux|     UK|  MEDIUM|   2|
|2026-01-01 00:54:51|    575|137.235.120.100|  FAILED_LOGIN|      Linux|     UK|     LOW|   3|
|2026-01-01 01:03:18|    274|179.188.226.165|PASSWORD_RESET|      Linux|  India|  MEDIUM|   4|
|2026-01-01 01:36:54|    214| 126.32.184.153|PASSWORD_RESET|      Linux|Germany|     LOW|   5|
|2026-01-01 01:38:06|    283| 37.247.113.192|FIREWALL_BLOCK|      MacOS| Canada|    HIGH|   6|
|2026-01-01 01:54:48|    928| 120.155.71.244| LOGIN_SUCCESS|    Windows|Germany|  MEDIUM|   7|
|2026-01-01 02:21:51|    201| 196.64.128.183|  FAI

In [61]:
df.groupBy('device_type').count().show()

+-----------+-----+
|device_type|count|
+-----------+-----+
|      Linux| 3300|
|      MacOS| 3349|
|    Windows| 3351|
+-----------+-----+

